In [1]:
import pandas as pd
from tqdm import tqdm

from dataset import MyriadLamaDataset

dataset = MyriadLamaDataset(model_name="llama3.1_3b_it")
dataloader = dataset.get_dataloader(batch_size=8, shuffle=False)

few_shot_context = dataset.get_few_shot_examples()
for uuids, answers, all_paraphrases in tqdm(dataloader, desc="Generating baseline (origin)"):
    # Use only the original questions (paraphrase0)
    original_questions = all_paraphrases[0]
    prompts = dataset.construct_prompts(few_shot_context, original_questions)
    break

/home/xzhao/softwares/anaconda3/envs/ensemble/lib/python3.13/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Dataset already exists at /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama/paraphrases_dataset. Loading from disk.


Generating baseline (origin):   0%|          | 0/250 [00:00<?, ?it/s]


In [2]:
from generate_myriadlama import get_few_shot_examples_with_paraphrases


few_shot_examples = get_few_shot_examples_with_paraphrases(dataset)

✅ FlexAttention is available


In [3]:
from generate_myriadlama import construct_prompt_new_format


for uuids, answers, all_paraphrases in tqdm(dataloader):
    batch_predictions = []
    batch_generations = []
    batch_templates = []

    # Process each question in batch
    for i, paraphrases in enumerate(zip(*all_paraphrases)):
        # All paraphrases in MyriadLAMA are manually generated
        # Simply select the first N paraphrases
        all_templates = list(paraphrases)
        selected_templates = all_templates[: 5]

        # Construct ONE prompt with ALL question paraphrases (NEW FORMAT)
        # Prompt has: instruction + few-shot examples + ALL main question paraphrases
        prompt = construct_prompt_new_format(
            dataset.instruction,
            few_shot_examples,
            selected_templates,  # Pass ALL paraphrases, not just one
        )
        break
    break

  0%|          | 0/250 [00:00<?, ?it/s]


In [4]:
print(prompt)

Based on the context, predict the [MASK] in the sentence in one word.

Q: [MASK] is the higher-level concept of adit.
Q: adit is also necessarily a [MASK].
Q: [MASK] is the superclass of adit.
A: tunnel

Q: Marty Ehrlich became known as a [MASK] player.
Q: Marty Ehrlich is a [MASK] player.
Q: Marty Ehrlich plays [MASK].
A: saxophone

Q: Officially, the people living in Basel use the language [MASK] for communication.
Q: [MASK] recognizes Basel as its official language.
Q: What language is officially spoken in Basel? [MASK].
A: German

Q: [MASK] consists of Mount Makiling.
Q: [MASK] encompasses Mount Makiling as its element.
Q: Mount Makiling represents a segment of [MASK].
A: Luzon

Q: Iowa and [MASK] are neighboring countries.
Q: You can go through Iowa to reach [MASK].
Q: Which country is adjacent to Iowa? [MASK].
A: Wisconsin

Q: This Is Your Life premiered on the network [MASK].
Q: [MASK] is the first air channel of This Is Your Life.
Q: Which service originally aired This Is Your 

In [5]:
from generate_myriadlama import concatenate_paraphrases_with_positions
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/net/tokyo100-10g/data/str01_01/xzhao/models/llama_hf/llama3.2_3b_it"
tokenizer = AutoTokenizer.from_pretrained(model_path)
concatenated_text, segment_positions, segment_metadata, original_length = (
    concatenate_paraphrases_with_positions(prompt, tokenizer)
)


In [6]:
print(concatenated_text)

Based on the context, predict the [MASK] in the sentence in one word.

Q: [MASK] is the higher-level concept of adit.

Q: adit is also necessarily a [MASK].

Q: [MASK] is the superclass of adit.

A: tunnel

Q: Marty Ehrlich became known as a [MASK] player.

Q: Marty Ehrlich is a [MASK] player.

Q: Marty Ehrlich plays [MASK].

A: saxophone

Q: Officially, the people living in Basel use the language [MASK] for communication.

Q: [MASK] recognizes Basel as its official language.

Q: What language is officially spoken in Basel? [MASK].

A: German

Q: [MASK] consists of Mount Makiling.

Q: [MASK] encompasses Mount Makiling as its element.

Q: Mount Makiling represents a segment of [MASK].

A: Luzon

Q: Iowa and [MASK] are neighboring countries.

Q: You can go through Iowa to reach [MASK].

Q: Which country is adjacent to Iowa? [MASK].

A: Wisconsin

Q: This Is Your Life premiered on the network [MASK].

Q: [MASK] is the first air channel of This Is Your Life.

Q: Which service originally ai

In [7]:
for pos, label in zip(segment_positions, segment_metadata):
    print(f"Position: {pos}, Label: {label}")

Position: (0, 17), Label: {'type': 'instruction', 'paraphrase_idx': None, 'few_shot_idx': None, 'fs_q_para_idx': None}
Position: (18, 32), Label: {'type': 'few_shot_q', 'paraphrase_idx': None, 'few_shot_idx': 0, 'fs_q_para_idx': 0}
Position: (33, 44), Label: {'type': 'few_shot_q', 'paraphrase_idx': None, 'few_shot_idx': 0, 'fs_q_para_idx': 1}
Position: (45, 57), Label: {'type': 'few_shot_q', 'paraphrase_idx': None, 'few_shot_idx': 0, 'fs_q_para_idx': 2}
Position: (58, 61), Label: {'type': 'few_shot_a', 'paraphrase_idx': None, 'few_shot_idx': 0, 'fs_q_para_idx': None}
Position: (62, 77), Label: {'type': 'few_shot_q', 'paraphrase_idx': None, 'few_shot_idx': 1, 'fs_q_para_idx': 0}
Position: (78, 91), Label: {'type': 'few_shot_q', 'paraphrase_idx': None, 'few_shot_idx': 1, 'fs_q_para_idx': 1}
Position: (92, 102), Label: {'type': 'few_shot_q', 'paraphrase_idx': None, 'few_shot_idx': 1, 'fs_q_para_idx': 2}
Position: (103, 107), Label: {'type': 'few_shot_a', 'paraphrase_idx': None, 'few_shot_

In [8]:
print(segment_positions)

[(0, 17), (18, 32), (33, 44), (45, 57), (58, 61), (62, 77), (78, 91), (92, 102), (103, 107), (108, 127), (128, 140), (141, 154), (155, 158), (159, 170), (171, 184), (185, 197), (198, 202), (203, 214), (215, 227), (228, 240), (241, 244), (245, 258), (259, 275), (276, 290), (291, 304), (305, 318)]


In [9]:
from generate_myriadlama import create_myriadlama_mask

mask_mod = create_myriadlama_mask(
        segment_positions, segment_metadata, original_length
)


In [10]:
from generate_myriadlama import FlexAttentionWrapper


model = AutoModelForCausalLM.from_pretrained(
    model_path, device_map="cuda:0", torch_dtype="auto"
)
flex_wrapper = FlexAttentionWrapper(model)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [11]:
flex_wrapper.patch_model(mask_mod)

In [12]:
model.model.layers[0].self_attn

LlamaAttention(
  (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
  (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
  (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
  (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
)

In [13]:
inputs = tokenizer(
    concatenated_text, return_tensors="pt", truncation=True, add_special_tokens=True
).to(model.device)


In [2]:
logits = model(inputs["input_ids"]).logits[:, -1, :]

NameError: name 'model' is not defined

In [ ]:
aa = BatchedTensor()